# Modelo de Identificação de Fraudes

- Informações da base:
    - 'Time' (tempo): contém os segundos decorridos entre cada transação e a primeira transação no conjunto de dados. 
    - 'Amount' (valor): é o valor da transação 
    - 'Class' (classe): é a variável de resposta e assume valor 1 em caso de fraude e 0 caso contrário.
    - "Infelizmente, devido a questões de confidencialidade, não podemos fornecer os recursos originais e mais informações básicas sobre os dados. Características V1, V2, … V28 são os principais componentes obtidos com PCA"

In [ ]:
# Configuração do ambiente

# Garante que o notebook está na raiz do projeto
%cd .. 

# Verifica o diretório atual (Linux/Mac)
# !pwd  

# Verifica o diretório atual (Windows)
!cd 

In [ ]:
# Importações necessárias 

import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime as dt
import xgboost as xgb

from scipy.stats import kstest, normaltest, anderson

from sklearn.preprocessing import PowerTransformer, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree, metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids, NearMiss 
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTEENN

from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,
)

import warnings
from src.config.paths import RAW_DATA_PATH, PROCESSED_DATA_PATH
from src.config.auxiliares_ml import downcast_dataframe, testar_modelos_com_undersampling

warnings.filterwarnings('ignore')
sns.set_theme(palette="bright")

In [ ]:
# Importando a base

transacoes = pd.read_parquet(PROCESSED_DATA_PATH)

In [ ]:
# Verificar se todos os valores podem ser convertidos para inteiro (sem valores decimais)

if (transacoes['Time'] == transacoes['Time'].astype(int)).all():
    
    # Se todos os valores são inteiros, realizar a conversão
    
    transacoes['Time'] = transacoes['Time'].astype(int)
   
else:
    print("Nem todos os valores na coluna 'Time' podem ser convertidos para inteiro.")

In [ ]:
# Visualizando as informações após alterar o tipo de 'Time'

transacoes.info()

In [ ]:
# Visualizando a estatística descritiva

transacoes.describe()

#### Observando o quanto o target 'Class' está em desequilíbrio

In [ ]:
# Aplicando o tema do Seaborn com fundo cinza claro

sns.set_theme(style="whitegrid", palette="bright")

# Contagem dos valores na coluna 'Class'

class_counts = transacoes['Class'].value_counts()
labels = class_counts.index
sizes = class_counts.values

# Definir cores personalizadas (azul para Class 0 e vermelho para Class 1)
# Mantendo o azul original para Class 0 e usando vermelho para Class 1

colors = ['#2563EB' if label == 0 else '#E63946' for label in labels]  # Azul para 0, vermelho para 1

# Criando o gráfico de pizza com Seaborn e Matplotlib

plt.figure(figsize=(6, 6))
wedges, texts, autotexts = plt.pie(
    sizes, labels=labels, autopct='', colors=colors,
    startangle=140, wedgeprops={'edgecolor': 'black', 'linewidth': 1.5}, 
    textprops={'fontsize': 14, 'weight': 'bold'}
)

# Criando a legenda com as porcentagens

percentages = [f"{sizes[i] / sum(sizes) * 100:.1f}%" for i in range(len(sizes))]
legend_labels = [f"Class {labels[i]} - {percentages[i]}" for i in range(len(labels))]
plt.legend(wedges, legend_labels, title="Legenda", loc="best", fontsize=9)

plt.title("Distribuição da Class", fontsize=16, fontweight="bold", color='#2563EB')
plt.show()

### Observando se há presença de outliers

In [ ]:
# Removendo 'Time' e 'Class', pois normalmente não são usados em análise de variáveis contínuas

features = [col for col in transacoes.columns if col not in ['Class']]

# Configuração do layout

fig, axes = plt.subplots(nrows=8, ncols=4, figsize=(16, 32))
fig.suptitle('Boxplots das Variáveis', fontsize=16)

# Converter axes para um array 1D para iteração

axes = axes.flatten()

# Criar os boxplots

for i, feature in enumerate(features):
    sns.boxplot(y=transacoes[feature], ax=axes[i])
    axes[i].set_title(feature)
    axes[i].set_xlabel('')

# Ajustar espaçamento

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

### Testes de Hipóteses

Para termos certeza se aplicaremos no pré-processamento **Padronização** e/ou **Normalização** e/ou **Regularização**, precisamos saber se as amostras possuem uma distribuição normal ou próximo a uma distribuição normal. 
Lembrando que um dos algoritmos que iremos testar é a LogisticRegression, que assim como a Regressão Linear, Máquinas de Vetores de Suporte (SVMs) e redes neurais, se beneficia de uma distribuição normal (**Z-score Z-score normalization**) ou algo próximo a uma distribuição normal.

In [ ]:
# Lista de colunas numéricas (excluindo "Time" e "Class", que é o target)

colunas_numericas = [col for col in transacoes.columns if col not in ['Time', 'Class']]

# Dicionário para armazenar os resultados

resultados = {}

for col in colunas_numericas:
    dados = transacoes[col].dropna()  # Remover valores NaN
    
    # Normalizar os dados (média 0, desvio padrão 1) para o teste KS
    
    dados_norm = (dados - dados.mean()) / dados.std()
    
    # Teste KS
    ks_p = kstest(dados_norm, 'norm')[1]  # P-valor
    
    # Teste de D’Agostino e Pearson
    
    dagostino_p = normaltest(dados)[1]
    
    # Teste de Anderson-Darling
    
    anderson_stat = anderson(dados, 'norm').statistic
    
    # Consideramos que p > 0.05 sugere normalidade (não rejeitamos H0)
    
    normal = (ks_p > 0.05) and (dagostino_p > 0.05)
    
    # Armazena os resultados
    
    resultados[col] = {
        'KS_p': ks_p,
        'D’Agostino_p': dagostino_p,
        'Anderson_Stat': anderson_stat,
        'Normal': normal
    }

# Transformar em DataFrame para melhor visualização

df_resultados = pd.DataFrame(resultados).T

# Exibir as colunas que têm maior probabilidade de serem normais

df_resultados.sort_values(by="Anderson_Stat")

#### Correlação:

**Vamos agora observar a correlação das features com o target ('Class').**

In [ ]:
# Calcular a correlação com o target

correlation = transacoes.corr()['Class'].drop('Class').sort_values(ascending=False)

# Exibir um gráfico de barras com as correlações

plt.figure(figsize=(12, 6))
sns.barplot(x=correlation.index, y=correlation.values, palette='coolwarm')
plt.xticks(rotation=90)
plt.title("Correlação das Features com a Variável Target ('Class')")
plt.show()

#### As melhores 20 features

**Vamos usar o SelectKBest para estipularmos quais as melhores features para o modelo.**

In [ ]:
# Separando as features e a variável alvo

X = transacoes.drop(columns=['Class'])
y = transacoes['Class']

# Aplicando SelectKBest para selecionar as 10 melhores features

selector = SelectKBest(score_func=f_classif, k=20)
X_new = selector.fit_transform(X, y)

# Criando um novo DataFrame com as melhores colunas

selected_features = X.columns[selector.get_support()]
X_best = pd.DataFrame(X_new, columns=selected_features, index=X.index)

# Adicionando a coluna 'Class' ao novo DataFrame

X_best['Class'] = y

print("Melhores Features:", selected_features)

### Pré-processamento: 

#### **NOTA**: Em princípio vamos aplicar algoritmos de ML sem balanceamento na base.

- Utilizaremos o **PowerTransformer** devido ao grande número de outliers, **exceto em 'Time'**. <br>
- Em **'Time'** utilizaremos o **MinMaxScaler**.

In [ ]:
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

# Criando os escaladores

power_transformer = PowerTransformer(method='yeo-johnson')  # Funciona para valores positivos e negativos
minmax_scaler = MinMaxScaler()

# Aplicando a normalização nas colunas específicas

transacoes['Amount'] = power_transformer.fit_transform(transacoes[['Amount']])
transacoes['Time'] = minmax_scaler.fit_transform(transacoes[['Time']])

transacoes.head(2)

In [ ]:
# Separando X e y

X = transacoes.drop(columns='Class')
y = transacoes['Class']

In [ ]:
# Separando em treino e teste

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0,stratify=y)

### XGBoost 

In [ ]:
# Criando o modelo XGBoost

modelo_xgb = xgb.XGBClassifier(
    max_depth=9,
    n_estimators=200,
    learning_rate=0.3,
    random_state=0,
    scale_pos_weight=(y.value_counts()[0] / y.value_counts()[1])  # Lida com desbalanceamento
)

# Treinando e avaliando o modelo

modelo_xgb.fit(X_train, y_train)
y_pred_xgb = modelo_xgb.predict(X_test)

# Calculando as métricas

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)

# Vusualizando a matriz confusão

confusion_matrix(y_test, y_pred_xgb)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_pred_xgb))
print('Precisão:', precision_score(y_test, y_pred_xgb))
print('Recall:', recall_score(y_test, y_pred_xgb, pos_label=1))

### Random Forest

In [ ]:
# Utilizando o Random Forest

clf_RF = RandomForestClassifier(
    max_depth=7,  
    random_state=0,
    criterion='gini',
    n_estimators=100,  
    class_weight='balanced_subsample' 
).fit(X_train, y_train)

y_pred_RF = clf_RF.predict(X_test)

# Vusualizando a matriz confusão

confusion_matrix(y_test, y_pred_RF)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_pred_RF))
print('Precisão:', precision_score(y_test, y_pred_RF))
print('Recall:', recall_score(y_test,y_pred_RF))

### Regressão Logística

In [ ]:
# Utilizando a Regressão Logística

clfLog = LogisticRegression(random_state=0).fit(X_train, y_train)

# Calculando as métricas

y_predLog = clfLog.predict(X_test)
y_pred_probaLog = clfLog.predict_proba(X_test)[:,1]

# Vusualizando a matriz confusão
confusion_matrix(y_test, y_predLog)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_predLog))
print('Precisão:', precision_score(y_test, y_predLog))
print('Recall:', recall_score(y_test, y_predLog))

### SVC

In [ ]:
# Utilizando o SVC

clfSVC = SVC(random_state=0, probability=True).fit(X_train, y_train)

# Fazendo a previsão

y_predSVC = clfSVC.predict(X_test)
y_pred_probaSVC = clfSVC.predict_proba(X_test)[:,1]

# Visualizando a matriz confusão

confusion_matrix(y_test, y_predSVC)

In [ ]:
# Utilizando as métricas de avaliação

print('Acurácia:', accuracy_score(y_test, y_predSVC))
print('Precisão:', precision_score(y_test, y_predSVC))
print('Recall:', recall_score(y_test, y_predSVC))

### OBSERVAÇÃO: 

Com a base desbalanceada, os três melhores resultados foram:<br>

- **1º Lugar: XGBoost**

    Acurácia: 0.9995531296881484 / Precisão: 0.9411176470588235 / Recall: 0.7901212345679012 

- **2º Lugar: Random Forest**

    Acurácia: 0.9992658559162437 / Precisão: 0.7751479289940828 / Recall: 0.808641975308642

- **3º Lugar: SVC**<br>

    Acurácia: 0.9993509740708821 / Precisão: 0.9719626168224299 / Recall: 0.6419753086419753534234